# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

# Personalized Technical Tutor

This tool takes a technical question and generates an easy to understand explanation using:

1. GPT-5-nano through OpenAI API
2. Open Source Model running locally through Ollama

Both models receive the same question and same tutor instructions.

In [ ]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


## Constants

It Stores the model names and local ollama endpoint.

GPT-5-nano will be called through OpenAI.

The open source model will be called locally through Ollama.

In [ ]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'


## Set Up the Enviornment

The OpenAI API Key is loaded from the .env file.

OpenAI client connects to GPT-5-nano.

Ollama connects to Open source model running locally.

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}

Please Explain:
- what is set comprehension does
- what book.get("author") does
- why the if condition is used
- what yield from does
- How duplicate authors are removed
- Why the order of the results may not be predictable.
"""

In [ ]:
# New Question

question = """
A binary classification model produced the following confusion matrix:

- True Positives = 40
- False Positives = 10
- True Negatives = 45
- False Negatives = 5

Please calculate and explain:

- Accuracy
- Precision
- Recall
- F1-score

Show each formula, substitute the values, and give the final answer rounded to 2 decimal places.
"""

## Messages

The same System Prompt and Technical question wil be sent to both models.

This makes it easier to compare the explanation.

In [ ]:
# Mesaages
messages = [
    {"role":"system", "content":"system_prompt"},
    {"role":"user", "content":question}
]

In [ ]:
# Get gpt-5-nano to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response),
    display_id = display_handle.display_id
    )

In [ ]:
# Warm up Llama 3.2 - It was taking a lot of time to run, just for warm up

warmup = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role":"user", "content":"Hi"}
    ]
)

print("Llama 3.2 is ready")

In [ ]:
# Get Llama 3.2 to answer
response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages
)

display(Markdown(response.choices[0].message.content))

# Week 2 Upgrade - Conversation Context

The Personalized Technical Tutor is now improved by giving it the previous conversations as background information.

This helps the tutor understand the context before answering the new question.

## Previous Conversations

The last two conversations are stored as background information.

The tutor will use them before answering the new technical question.

In [ ]:
# Previous Conversations

previous_conversation_1 = """
User: What is supervised learning?
Tutor: Supervised learning is a type of machine learning where a model learns from labeled data.
"""

previous_conversation_2 = """
User: What is overfitting?
Tutor: Overfitting happens when a model learns the training data too closely and does not perform well on new data.
"""

In [ ]:
# New Question: Now a new technical question is asked based on the previous conversations.
question = """
How can I prevent overfitting when training a supervised machine learning model?

Please Explain:

- Why overfitting happens
- How train and validation data helps
- How regularization helps
- How cross validation helps
- How early stopping helps
"""

## User Prompt

The previous conversations and the new technical question are combined into one User Prompt.

This gives the tutor the conversation context before answering.

In [ ]:
# User Prompt

user_prompt = f"""
Previous Conversation 1:
{previous_conversation_1}

Previous Conversation 2:
{previous_conversation_2}

New Technical Question:
{question}
"""

## System Prompt

The System Prompt defines how the Technical Tutor should explain the questions.

The same instructions will be used for both models.

In [ ]:
system_prompt = """
You are a technical tutor for Data Science and AI.

Explain technical concepts in simple and clear steps.

Assume familiarity with Python, statistics, data analysis and machine learning.

When helpful:
- Explain the main idea first
- Break the answer into simple steps
- Use practical examples
- Explain formulas and calculations clearly
"""

## Messages

The same System Prompt and User Prompt will be sent to both models.

The User Prompt now contains the previous conversations and the new technical question.

In [ ]:
# Messages

messages = [
    {"role":"system", "content":system_prompt},
    {"role":"user", "content":user_prompt}
]

In [ ]:
# Get gpt-5-nano to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response),
    display_id = display_handle.display_id
    )

In [ ]:
# Reusing Llama 3.2 to answer, with streaming
stream = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(
        Markdown(response),
        display_id=display_handle.display_id
    )